In [1]:
import json
import random
import uuid
from datetime import datetime, timedelta

import mysql.connector
from faker import Faker
from pyspark.sql import SparkSession

In [2]:
def generate_event_logs(fake: Faker, user_ids: list[int]):
    event_types = [("USER_SIGNUP", "USER"),
                   ("USER_LOGIN", "USER"),
                   ("ORDER_CREATED", "ORDER"),
                   ("ORDER_CANCELLED", "ORDER"),
                   ("PAYMENT_REQUESTED", "PAYMENT"),
                   ("PAYMENT_APPROVED", "PAYMENT"),
                   ("PAYMENT_FAILED", "PAYMENT"),
                   ("SHIPMENT_CREATED", "SHIPMENT"),
                   ("SHIPMENT_DELIVERED", "SHIPMENT")]
    event_type, entity_type = random.choice(event_types)
    return {
        "event_uuid": str(uuid.uuid4()),
        "event_time": datetime.now() - timedelta(seconds=random.randint(0, 86400)),
        "event_type": event_type,
        "entity_type": entity_type,
        "entity_id": random.randint(1, 10_000),
        "user_id": random.choice(user_ids),
        "payload": {"msg": fake.text(max_nb_chars=120)}
    }


def get_connection(config):
    return mysql.connector.connect(
        host=config.get("host", "localhost"),
        port=int(config.get("port", 3306)),
        user=config.get("user", "root"),
        password=config.get("password", ""),
        database=config.get("database", ""),
        charset=config.get("charset", "utf8mb4"),
        connection_timeout=int(config.get("connect_timeout", 5)),
        autocommit=False,
        use_pure=True
    )


def partition_insert(config, batch_size_=5000):
    sql = "INSERT INTO event_log (event_uuid, event_time, event_type, entity_type, entity_id, user_id, payload) VALUES (%s,%s,%s,%s,%s,%s,%s)"

    def _writer(rows_iter):
        conn = get_connection(config)
        cur = conn.cursor()
        batch = []
        try:
            for r in rows_iter:
                if isinstance(r, dict):
                    event_uuid = r["event_uuid"]
                    event_time = r["event_time"]
                    event_type = r["event_type"]
                    entity_type = r["entity_type"]
                    entity_id = int(r["entity_id"])
                    user_id = (int(r["user_id"]) if r.get("user_id") is not None else None)
                    payload = r.get("payload")
                else:
                    event_uuid = r["event_uuid"]
                    event_time = r["event_time"]
                    event_type = r["event_type"]
                    entity_type = r["entity_type"]
                    entity_id = int(r["entity_id"])
                    user_id = (int(r["user_id"]) if r["user_id"] is not None else None);
                    payload = r["payload"]

                payload_str = payload if isinstance(payload, str) else json.dumps(payload, ensure_ascii=False) if payload is not None else None
                batch.append((event_uuid, event_time, event_type, entity_type, entity_id, user_id, payload_str))

                if len(batch) >= batch_size_:
                    cur.executemany(sql, batch)
                    conn.commit()
                    batch.clear()

            if batch:
                cur.executemany(sql, batch)
                conn.commit()
                batch.clear()

        except Exception:
            conn.rollback()
            raise
        finally:
            try:
                cur.close()
            except Exception:
                pass
            conn.close()

    return _writer

In [3]:
host = "mysql-primary.mmix.io"
port = 3306
database = "mmix"
user = "mmix"
password = "mmix"
user_table = "users"
event_log_table = "event_logs"
driver = "com.mysql.cj.jdbc.Driver"
jdbc_url = f"jdbc:mysql://{host}:{port}/{database}?useUnicode=true&characterEncoding=utf8&useSSL=false&allowPublicKeyRetrieval=true"
properties = {"host": host, "port": str(port), "user": user, "password": password, "database": database, "driver": driver}
total = 10000
batch_size = 1000
num_partitions = 8

In [4]:
spark = SparkSession.builder.appName("Mysql Example").master("spark://spark-master.mmix.io:7077").getOrCreate()
users = spark.read.jdbc(url=jdbc_url, table=user_table, properties=properties)
users_ids = [int(row.user_id) for row in users.select("user_id").collect()]

26/01/23 17:55:01 WARN Utils: Your hostname, MacBook-Pro-14.local resolves to a loopback address: 127.0.0.1; using 10.12.2.32 instead (on interface en0)
26/01/23 17:55:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/23 17:55:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/23 17:55:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/23 17:55:01 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/01/23 17:55:01 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/01/23 17:55:01 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/01/23 17:55:01 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting po

### MapPartitions & UFD 이용

In [5]:
event_logs = [generate_event_logs(Faker("ko_KR"), users_ids) for _ in range(total)]
spark.createDataFrame(event_logs).rdd.repartition(num_partitions).foreachPartition(partition_insert(properties, batch_size))

IndexError: list index out of range

In [ ]:
spark.stop()